<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_87_graphrag_knowledge_graphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🕸️ **Module 87 — GraphRAG + Knowledge Graphs** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 12 · Production Extensions


# Module 87 — GraphRAG + Knowledge Graphs

> **M30 vanilla RAG** retrieves the top-`k` semantically similar chunks for a query. That works great for "summarise this PDF" but **silently fails** on multi-hop questions: *"Which engineers worked on the Q3 reorg AND have approved budget authority for the next fiscal year?"* No single chunk contains both facts. **GraphRAG** — Microsoft's 2024 paper and the open-source movement that followed — fixes this by building a **knowledge graph** from your corpus and traversing it at query time. Plus structured citations, plus multi-hop reasoning, plus better-tasting "executive summary" outputs.
>
> This module: the graph primer you need, the Microsoft GraphRAG recipe, the open-source alternatives, and when to reach for graph + vector hybrid retrieval vs plain vector RAG.

### What you'll cover
1. Why vanilla RAG fails on multi-hop + global-summary questions
2. Knowledge-graph primer — nodes, edges, ontologies, RDF vs property graphs
3. **Neo4j + Cypher** essentials in 10 queries
4. **LLM-driven graph construction** — extracting entities + relations from raw text
5. **Microsoft GraphRAG (2024)** — communities, hierarchical summaries, the **global vs local** query split
6. **LightRAG / nano-graphrag / GraphRAG.ai** — the open-source variants
7. **Hybrid graph + vector retrieval** — the production pattern
8. **LLM-generated Cypher** for direct querying (LangChain `GraphCypherQAChain`)
9. The adjacent ideas — **HippoRAG · RAPTOR · ReadAgent · KG-RAG**
10. Production graph stack: **Neo4j · Memgraph · Kùzu · NebulaGraph · TigerGraph · Amazon Neptune**
11. When to pick GraphRAG vs vanilla RAG vs SQL vs ColPali


## 1 · Why vanilla RAG fails on the hard questions

The M30 setup: chunk the corpus, embed, store in a vector DB (M42), retrieve top-`k` by cosine similarity, stuff into the prompt. Three failure modes you'll hit at scale:

| Question shape | Why vanilla RAG breaks |
|---|---|
| **Multi-hop** — "which projects did engineers in the Search team ship that the CFO approved?" | the answer requires joining ≥ 2 facts that live in *different* chunks; cosine alone can't traverse the join |
| **Global summary** — "give me the top 5 themes in the entire codebase reorganisation programme this year" | top-`k` chunks miss the *whole-corpus* signal — you'd need to read everything |
| **Diversity / coverage** — "list ALL the customers who reported this bug" | embedding-similarity clusters around the closest match; you miss the long tail |
| **Reasoning about structure** — "is this person a manager of X who reports to Y?" | corpus structure isn't a sequence of independent chunks; it's a *graph* |
| **Time-evolution** — "what was our pricing in Q2 vs Q3 vs Q4?" | chunks have no temporal join; vector search ignores time |

Microsoft's 2024 GraphRAG paper pinned the term: **"global sense-making" queries** (entire-corpus summaries + multi-hop reasoning) are where vanilla RAG falls over hardest.

A **knowledge graph** captures the relations between entities directly. Once you have the graph, the join is free.

## 2 · Knowledge-graph primer

A knowledge graph (KG) is **nodes** (entities) connected by **edges** (relations). Two big traditions:

| Tradition | Standards / tooling | What dominates today |
|---|---|---|
| **RDF (Resource Description Framework)** | W3C; triples `(subject, predicate, object)`; **SPARQL** query language; **OWL** ontology language | Wikidata, DBpedia, scientific KGs, the open semantic web |
| **Property graph** | nodes + edges, each with arbitrary key-value properties; **Cypher** + **Gremlin** query languages | Neo4j, Memgraph, Amazon Neptune, NebulaGraph — the production-data world |

The 2026 honest take: **property graphs dominate production AI**. Cypher is easier to read; Neo4j has the biggest community + LLM-integration ecosystem. RDF is what you reach for if you must interoperate with academic / government / pharmaceutical KGs.

### Vocabulary you need
| Term | Meaning |
|---|---|
| **Node** (vertex) | an entity — `Person`, `Document`, `Project`, `Customer` |
| **Edge** (relationship) | a typed connection — `(:Person)-[:MANAGES]->(:Person)` |
| **Property** | key-value pair on a node or edge — `name: "Alice"`, `since: 2023` |
| **Label** | the type of a node — `:Person` |
| **Ontology / schema** | what node + edge types are allowed and how |
| **Triple** (RDF) | `(s, p, o)` — three entities |
| **Cypher** | Neo4j's pattern-matching query language |


## 3 · Neo4j + Cypher essentials

Cypher reads like ASCII art of a graph: `()` for nodes, `-[]->` for edges. **Ten queries** that cover 90% of what you'll write.

In [ ]:
# Spin up Neo4j locally:  docker run -p 7687:7687 -p 7474:7474 -e NEO4J_AUTH=none neo4j
# Then connect with the official driver:

neo4j_basics = '''
# pip install neo4j

from neo4j import GraphDatabase
driver = GraphDatabase.driver("bolt://localhost:7687", auth=None)

with driver.session() as s:
    # 1. CREATE nodes + a relationship
    s.run(r'''
      CREATE (a:Person {name: "Alice", title: "VP Engineering"})
      CREATE (b:Person {name: "Bob",   title: "Engineer"})
      CREATE (a)-[:MANAGES {since: 2023}]->(b)
    ''')

    # 2. MATCH (read pattern) — who does Alice manage?
    rows = s.run(r'''
      MATCH (a:Person {name: "Alice"})-[:MANAGES]->(b)
      RETURN b.name AS report
    ''').data()
    print(rows)   # [{'report': 'Bob'}]

    # 3. MERGE = upsert (create if missing, match otherwise)
    s.run("MERGE (p:Person {name: $name}) RETURN p", name="Carol")

    # 4. Properties + labels filter
    s.run(r'''
      MATCH (e:Person {title: "Engineer"})
      RETURN e.name
    ''')

    # 5. 2-HOP traversal — "who manages people who report to Alice?"
    s.run(r'''
      MATCH (a:Person {name: "Alice"})-[:MANAGES]->(:Person)<-[:MANAGES]-(boss)
      RETURN boss.name
    ''')

    # 6. SHORTEST PATH between two nodes
    s.run(r'''
      MATCH p = shortestPath(
        (a:Person {name: "Alice"})-[*..6]-(c:Person {name: "Carol"})
      )
      RETURN [n IN nodes(p) | n.name] AS path
    ''')

    # 7. AGGREGATION — count of reports per manager
    s.run(r'''
      MATCH (m:Person)-[:MANAGES]->(r:Person)
      RETURN m.name, count(r) AS direct_reports
      ORDER BY direct_reports DESC
    ''')

    # 8. Add an index for fast lookup (production essential)
    s.run("CREATE INDEX person_name IF NOT EXISTS FOR (p:Person) ON (p.name)")

    # 9. Vector property + index (Neo4j 5.11+) — for graph + vector hybrid
    s.run(r'''
      CALL db.index.vector.createNodeIndex(
        'doc_embeddings', 'Document', 'embedding', 1536, 'cosine')
    ''')

    # 10. Cypher's CALL ... YIELD — invoke procedures (graph algorithms via GDS)
    s.run(r'''
      CALL gds.pageRank.stream({
        nodeProjection: 'Person',
        relationshipProjection: 'MANAGES'
      })
      YIELD nodeId, score
      RETURN gds.util.asNode(nodeId).name AS person, score
      ORDER BY score DESC LIMIT 10
    ''')
'''
print(neo4j_basics)

**Three Cypher patterns to internalise:**
1. **`MATCH (a)-[:R]->(b)`** — pattern matching is *the* primitive; everything else is sugar.
2. **`MERGE`** — atomic upsert; use it on every entity write or you get duplicates.
3. **`OPTIONAL MATCH`** — like LEFT JOIN; matches when possible, returns NULL when not.

Production rule: **always create an index on the property you `MATCH` by** (`CREATE INDEX person_name FOR (p:Person) ON (p.name)`) — without it, every query is a full scan.

## 4 · LLM-driven graph construction

You rarely *have* a clean KG. You usually have **unstructured documents**. The 2024+ recipe: use an LLM to extract `(entity, relation, entity)` triples from raw text.

In [ ]:
# Pattern: LLM-as-extractor. Pseudocode using LangChain's LLMGraphTransformer.
graph_construction = '''
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI
from langchain_community.graphs import Neo4jGraph
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
graph = Neo4jGraph(url="bolt://localhost:7687", username="neo4j", password="...")

# 1. Build the transformer — optionally constrain to an ontology
transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Project", "Document", "Company"],
    allowed_relationships=["MANAGES", "AUTHORED", "MENTIONS", "WORKS_AT"],
    strict_mode=True,                              # reject anything not in the allowed lists
)

# 2. Extract triples from raw text
docs = [Document(page_content='''
   Alice (VP Engineering at Acme) leads the Q3 reorg project. The plan, authored by
   Bob and Carol, will be reviewed by the CFO in October. The Acme CTO mentioned
   the project in last week all-hands.
''')]
graph_docs = transformer.convert_to_graph_documents(docs)
graph.add_graph_documents(graph_docs, baseEntityLabel=True, include_source=True)

# Result in Neo4j:
#   (:Person {id:"Alice"})-[:MANAGES]->(:Project {id:"Q3 reorg"})
#   (:Person {id:"Bob"})-[:AUTHORED]->(:Project {id:"Q3 reorg plan"})
#   (:Company {id:"Acme"})-[:EMPLOYS]->(:Person {id:"Alice"})
#   ...
'''
print(graph_construction)

**Five common pitfalls** and how to fix them:

| Pitfall | Fix |
|---|---|
| **Entity duplication** ("Alice" vs "Alice Smith" vs "VP Engineering Alice") | Run an **entity-resolution** pass over extracted nodes — embedding similarity + heuristics, or use Neo4j's `node similarity` |
| **Schema drift** — different runs invent new relationship types | `strict_mode=True` + a fixed `allowed_relationships` list |
| **Hallucinated edges** | post-filter: drop any triple not explicitly stated; use stronger model for extraction |
| **Long-document context loss** | chunk + extract per chunk + dedupe globally |
| **Slow extraction at corpus scale** | use a **cheap model** (Haiku / GPT-mini / local Qwen) for extraction, **stronger model** for query-time reasoning |

The 2026 production recipe: cheap-model extraction → entity-resolution post-process → write to Neo4j with `MERGE` (idempotent) → schedule incremental re-runs only on changed docs.

## 5 · Microsoft GraphRAG (2024) — the recipe that started the term

[Microsoft Research, July 2024](https://arxiv.org/abs/2404.16130). Pipeline:

```
   1.  Chunk documents
                │
                ▼
   2.  LLM extracts entities + relationships per chunk
                │
                ▼
   3.  Build the graph (nodes, edges with multiple "claims" each)
                │
                ▼
   4.  COMMUNITY DETECTION (Leiden algorithm) over the graph
                │
                ▼
   5.  Per-community: LLM writes a HIERARCHICAL SUMMARY
       (recursively rolled up — level 0, 1, 2 ...)
                │
                ▼
   6.  Index summaries (and raw text) for retrieval
                │
                ▼
   7.  At query time:
       - LOCAL question → use entity neighbourhood + chunks
       - GLOBAL question → map over community summaries, reduce
```

### The two query modes

| Mode | When | How |
|---|---|---|
| **Local** | specific entities mentioned | walk the entity neighbourhood; retrieve linked chunks |
| **Global** | whole-corpus questions ("themes," "patterns") | map LLM over community summaries; reduce into final answer |

**The killer result**: on global sense-making questions, GraphRAG produces answers humans rate as more comprehensive than vanilla RAG, at comparable cost (the graph + summaries are built **once** at index time; queries are cheap).

## 6 · LightRAG, nano-graphrag, and the OSS GraphRAG explosion

The Microsoft paper was open-sourced, but slow and expensive. Five open-source variants you'll meet:

| Project | Notes |
|---|---|
| **Microsoft GraphRAG** | the canonical reference; Python; PostgreSQL or in-memory storage; LLM-heavy |
| **nano-graphrag** | ~1000-line minimal reimplementation; great for learning + customisation |
| **LightRAG** (HKU 2024) | dual-level retrieval (low: entities, high: themes); ~half the LLM calls of Microsoft's |
| **GraphRAG.ai** | hosted SaaS by ex-MSR team |
| **HiRAG** | hierarchical retrieval with explicit reasoning over the graph |
| **OpenRAG / KG-RAG** | grab bags of community-driven variants |
| **LangChain `GraphCypherQAChain`** | the simplest "LLM writes Cypher → run on Neo4j → LLM reads result" loop — for *local* questions |

The 2026 honest take: **start with LangChain's `Neo4jGraph` + `GraphCypherQAChain`** for proof of concept. Move to **LightRAG or nano-graphrag** when you need community summaries / global queries. Reach for **Microsoft GraphRAG** when you have the LLM budget for the reference recipe.

## 7 · Hybrid graph + vector retrieval — the production pattern

Most production stacks **combine** vector RAG (M30) with KG retrieval. The pattern:

```
                ┌─── classify query intent ────┐
                │                              │
                ▼                              ▼
        "local fact" query             "multi-hop / global" query
                │                              │
                ▼                              ▼
       Vector search (M42)          KG traversal (Cypher)
       top-k chunks                 entity ego-network + community summary
                │                              │
                └─────────────┬────────────────┘
                              ▼
                  Reranker / fusion (Cohere rerank / RRF)
                              │
                              ▼
                       LLM generation
                       (with citations)
```

Three fusion patterns:

| Fusion | When |
|---|---|
| **Reciprocal Rank Fusion (RRF)** | simple, cheap; mix two ranked lists |
| **Cross-encoder rerank** (Cohere / BGE) | best quality; 50-100ms per rerank |
| **LLM-as-reranker** | strongest but slow; use only when others fail |

Neo4j's **vector index** (since 5.11) lets you store the embedding **on the node itself** and combine `MATCH` + cosine search in a single Cypher query:

```cypher
CALL db.index.vector.queryNodes('doc_embeddings', 5, $query_vec) YIELD node, score
MATCH (node:Document)<-[:MENTIONS]-(p:Person)
WHERE p.team = 'Search'
RETURN node.title, p.name, score
```

That's **vector + graph + filter** in five lines — exactly what a hybrid retriever needs.

## 8 · LLM-generated Cypher — text-to-graph-query

For questions whose answer is *just a graph query* (not a paragraph), let the LLM **write the Cypher**, execute it, and summarise the result.

In [ ]:
text_to_cypher = '''
from langchain_neo4j import GraphCypherQAChain
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI

graph = Neo4jGraph(url="bolt://localhost:7687", username="neo4j", password="...")
graph.refresh_schema()   # introspect labels + relationship types

chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=False,        # disable arbitrary writes
)

ans = chain.invoke({"query": "Which engineers report (directly or indirectly) to Alice?"})
print(ans["result"])
# Behind the scenes the LLM generated something like:
#   MATCH (a:Person {name: "Alice"})-[:MANAGES*1..3]->(p:Person)
#   RETURN p.name
'''
print(text_to_cypher)

**Production guards for LLM-generated Cypher:**
1. **Read-only by default** — `allow_dangerous_requests=False` blocks `CREATE/DELETE/MERGE`.
2. **Schema injection** — give the LLM the live schema (`refresh_schema()`) so it generates valid labels + relationship types.
3. **Cypher validation** — parse the LLM's output before executing (Neo4j EXPLAIN; catches syntax errors cheaply).
4. **Query timeout + row cap** — `LIMIT 200`; abort runaway joins.
5. **Audit log** — every executed Cypher → SIEM with `tenant_id` (M51 / M80).

LLM-generated Cypher is one of the few text-to-X tools that actually works in production today, *because the query language is tightly constrained.*

## 9 · Adjacent ideas worth knowing

| Idea | What it adds |
|---|---|
| **HippoRAG** (Stanford 2024) | hippocampus-inspired memory; one-pass indexing with Personalised PageRank for retrieval |
| **RAPTOR** | recursive tree of summaries; alternative to graph + community summaries |
| **ReadAgent** | LLM agent that reads + gists + jumps through long docs |
| **GraphReader** | converts the document into a graph, then the LLM "walks" it as an agent |
| **Property-graph + ontology-driven RAG** | constrain the graph schema → far better extraction quality |
| **KG-augmented chain-of-thought** | inject KG triples into the reasoning trace at every CoT step |
| **Temporal Knowledge Graphs** (T-KG) | edges have time intervals — supports the "Q2 vs Q3" pricing question |
| **Time-aware GraphRAG** | summaries indexed by time window |

If your domain has strong existing ontologies (biology / pharma / finance / legal), you'll save 80% of construction time by **reusing the ontology** rather than letting the LLM invent its own.

## 10 · Production graph stack

| Engine | Strengths | Watch for |
|---|---|---|
| **Neo4j AuraDB / Community / Enterprise** | biggest ecosystem; **vector index built-in** since 5.11; **GDS** library for graph algorithms | Enterprise license cost |
| **Memgraph** | in-memory, fast for streaming graphs; Cypher-compatible | smaller community |
| **Kùzu** | embedded, columnar (DuckDB-style); excellent for analytics + LLM workloads | younger; fewer integrations |
| **NebulaGraph** | distributed, very large graphs | steep ops learning curve |
| **TigerGraph** | enterprise distributed; GSQL language | proprietary; enterprise-only |
| **Amazon Neptune** | AWS-managed; supports both property + RDF | AWS-only; smaller community |
| **Dgraph** | distributed, GraphQL-native | smaller community in 2025 |
| **ArangoDB** | multi-model (graph + doc + KV) | jack-of-all-trades |
| **FalkorDB (formerly RedisGraph)** | redis-module graph DB | Redis ecosystem |
| **Apache AGE (Postgres extension)** | graph queries on top of Postgres (M36) | best when you already have Postgres |

The 2026 default for AI: **Neo4j AuraDB** (managed) or **Kùzu** (embedded). Both have first-class LangChain + LlamaIndex integrations and built-in vector indexes.

## 11 · When to pick what — the decision tree

```
   Is the question a JOIN or MULTI-HOP across entities?           ► YES → GRAPH RAG
                                                                   ► NO  ──┐
   Does the answer require reading a SPECIFIC document chunk?     ► YES → VANILLA RAG (M30)
                                                                   ► NO  ──┐
   Is the question STRUCTURED + over tabular data?                ► YES → SQL ON A WAREHOUSE / pgvector (M36)
                                                                   ► NO  ──┐
   Is the corpus mostly LAYOUT-RICH PDFs (slides, financial)?     ► YES → ColPali / late-interaction (M65 §8)
                                                                   ► NO  ──┐
   Is the question a GLOBAL SUMMARY of the whole corpus?          ► YES → GRAPH RAG with community summaries
                                                                   ► NO  → VANILLA RAG (M30) + reranker
```

### Anti-patterns
- ❌ **Building a graph for everything.** Most questions are local — vanilla RAG is cheaper and works fine.
- ❌ **No entity resolution.** "Alice" + "Alice Smith" + "Alice S." as three nodes destroys joins.
- ❌ **Free-form LLM extraction with no ontology.** Every run invents new relationship types; the graph becomes useless.
- ❌ **No index on the property you `MATCH` on.** Cypher gives you SQL-like performance ONLY with the right indexes.
- ❌ **Generating Cypher with no read-only guard.** A leaked LLM key can `DROP DATABASE` if you forget.
- ❌ **Rebuilding the graph from scratch every run.** Incremental updates with `MERGE` are non-negotiable past 100K nodes.

## ✅ Recap
- Vanilla RAG fails on **multi-hop**, **global summaries**, **structure**, and **time-evolution**. A KG fixes those.
- **Property graphs (Neo4j + Cypher)** dominate production AI; RDF dominates the open semantic web.
- Build the KG with **LLM-driven extraction** + **strict ontology** + entity resolution.
- **Microsoft GraphRAG** introduced the community-summary recipe with **local vs global** query modes.
- **OSS variants** (LightRAG, nano-graphrag, HiRAG) give you 80% of the recipe at 10% of the LLM bill.
- The production pattern is **graph + vector hybrid** with RRF / cross-encoder rerank fusion.
- **LLM-generated Cypher** works in production *because the language is constrained* — guard it with read-only + schema injection + timeout.
- Pick **Neo4j AuraDB** or **Kùzu** for new builds. **Apache AGE** if you live on Postgres.

Next: **M88 — Synthetic Data Generation + Distillation**.
